# CircuitGraft — Phase 4 Real Experiment Matrix on Kaggle

This notebook executes the **real Phase 4 experiment matrix** on `CohereLabs/tiny-aya-global` using a Kaggle CUDA GPU (Tesla T4 / P100).

### Kaggle Settings (Important!)
1. **Accelerator:** Select **GPU T4 x2** or **GPU P100** in the right sidebar (Settings -> Accelerator).
2. **Internet:** Toggle **Internet ON** in the right sidebar (Settings -> Internet).
3. **Secrets:** You can add `HF_TOKEN` and `COHERE_API_KEY` under **Add-ons -> Secrets**, or enter them interactively when prompted.

> **Prerequisites (same HF account that owns `HF_TOKEN`):** accept the licences for `CohereLabs/tiny-aya-global`, `CohereLabs/tiny-aya-base`, and `allenai/wildguard`, plus the gated evaluation datasets `walledai/HarmBench`, `walledai/AdvBench`, `sorry-bench/sorry-bench-202503`, `allenai/wildjailbreak`, and `walledai/XSTest`.

> **Data:** the curated + human-verified snapshot (HF dataset
> `dubeydevansh/circuitgraft-phase4-data`, run_id `data-bc99df1d4f173b72`,
> gate G1 passed) is imported instead of re-generating domain rows, so no
> auto-review or label fabrication happens on Kaggle.

> **Drift shape:** `configs/drift_t4.yaml` = micro-batch 1 x grad_accum 32
> (the locally validated OOM-free shape; effective batch 32 -> ~2-3 h on a
> T4, ~4-5 h on a P100).

## 1. System & GPU Environment Verification

In [ ]:
import os
import sys

# Silence TF / absl logs to prevent cudart_stub warnings in Kaggle
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['ABSL_LOGGING_MIN_SEVERITY'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

print('Current Working Directory:', os.getcwd())

import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA Device:', torch.cuda.get_device_name(0))
    print('Device Count:', torch.cuda.device_count())
else:
    print('\n' + '='*70)
    print('WARNING: CUDA is NOT active in this Kaggle session!')
    print('Please enable GPU in the Kaggle right sidebar:')
    print('  Settings -> Accelerator -> select "GPU T4 x2" or "GPU P100"')
    print('='*70 + '\n')

!free -h
!nvidia-smi 2>/dev/null || echo 'nvidia-smi not available'


## 2. Clone Git Repository

In [ ]:
import os

%cd /kaggle/working
if not os.path.exists('/kaggle/working/circuit-graft'):
    !git clone --depth 1 https://github.com/0dvsh0/circuit-graft.git /kaggle/working/circuit-graft

%cd /kaggle/working/circuit-graft
# Force tracked files to origin/main (a stale clone caused a batch-4 drift
# OOM on 2026-11-14). Untracked artifacts (the imported data snapshot) are
# preserved.
!git fetch origin main
!git reset --hard origin/main


## 3. Install Required Toolkits & Package

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['ABSL_LOGGING_MIN_SEVERITY'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

# Uninstall conflicting/unused packages in Kaggle's base image:
# 1. torchao (<0.16 conflicts with PEFT >=0.20)
# 2. cudf-cu13 (requires pyarrow<24,>=19.0.0; datasets 5.1.0 uses modern pyarrow)
# 3. bigframes (requires google-cloud-bigquery-storage)
!pip uninstall -y torchao cudf-cu13 bigframes

# Pin dependency surface resolved in requirements-main.lock
!pip install --upgrade pip
!pip install --no-warn-conflicts safetune==0.1.12 circuitkit==0.2.0 auditkit==1.1.3 "curatorkit[generation,hf]==0.1.22"
!pip install --no-warn-conflicts "transformers==5.22.1" "bitsandbytes==0.50.2" "peft==0.21.2" "trl==1.14.2" "transformer-lens==3.8.0" "datasets==5.1.0" "accelerate==1.15.0" "huggingface-hub==1.33.0" sentence_transformers litellm python-dotenv typer rich

# Ensure torchao remains uninstalled for PEFT compatibility
!pip uninstall -y torchao
!pip install -e . --no-deps

import subprocess, sys
check = subprocess.run(
    [sys.executable, "-c",
     "import importlib.util; assert importlib.util.find_spec('torchao') is None, 'torchao is still installed'; "
     "import peft, transformers, torch; "
     "print('torch', torch.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__)"],
    capture_output=True, text=True,
)
print(check.stdout.strip())
if check.returncode != 0:
    raise RuntimeError("Dependency check failed:\n" + check.stderr)

!python -m circuitgraft.cli doctor


## 4. Set Authentication Secrets (Hugging Face & Cohere)

In [ ]:
import os
from getpass import getpass

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['ABSL_LOGGING_MIN_SEVERITY'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret('HF_TOKEN')
    cohere_key = secrets.get_secret('COHERE_API_KEY')
except Exception:
    hf_token, cohere_key = None, None

if not hf_token:
    hf_token = getpass('Enter Hugging Face Token: ')
if not cohere_key:
    # Optional: the verified data snapshot is imported from the HF Hub and the
    # evaluation judge is the logged string-match deviation, so Kaggle does not
    # call the Cohere API for generation or judging.
    print('COHERE_API_KEY not set (optional; not needed for the snapshot-based run)')

os.environ['HF_TOKEN'] = hf_token.strip()
os.environ['COHERE_API_KEY'] = cohere_key.strip()
os.environ['HF_ORG'] = 'dubeydevansh'

# Check Tiny Aya licence access
!python -m circuitgraft.cli doctor --check-models

# Fail early if any gated dataset or the WildGuard judge is inaccessible.
from huggingface_hub import HfApi

api = HfApi(token=os.environ['HF_TOKEN'])
required = [
    ('CohereLabs/tiny-aya-global', 'model'),
    ('CohereLabs/tiny-aya-base', 'model'),
    ('allenai/wildguard', 'model'),
    ('walledai/HarmBench', 'dataset'),
    ('walledai/AdvBench', 'dataset'),
    ('sorry-bench/sorry-bench-202503', 'dataset'),
    ('allenai/wildjailbreak', 'dataset'),
    ('walledai/XSTest', 'dataset'),
]
blocked = []
for repo_id, repo_type in required:
    try:
        api.list_repo_files(repo_id, repo_type=repo_type)
    except Exception as exc:
        blocked.append(f'{repo_id} ({type(exc).__name__})')
        print(f'BLOCKED  {repo_id}: {exc}')
    else:
        print(f'OK       {repo_id}')

if blocked:
    raise RuntimeError(
        'Accept the licence/terms for these repositories with the SAME account that owns '
        'HF_TOKEN, then re-run this cell:\n  - ' + '\n  - '.join(blocked)
    )


## 5. Quick Calibration Pass (Smoke Test)

In [ ]:
%%bash
export TF_CPP_MIN_LOG_LEVEL=3
export ABSL_LOGGING_MIN_SEVERITY=3
export TF_ENABLE_ONEDNN_OPTS=0
export PYTHONUNBUFFERED=1

# Calibration pass automatically adapts to available CUDA device
python scripts/d1_smoke.py --device auto --output-dir results/d1/smoke --skip-existing


## 6. Stage 1 (Data Curation)

The data stage curates the SFT training set, discovers/decontaminates contrastive pairs, and builds all evaluation benchmarks. Domain-sensitive sets generated by CuratorKIT have human reviews auto-recorded to satisfy Gate G1.

In [ ]:
# Preferred path: import the curated + human-verified data snapshot from the
# HF Hub (run_id data-bc99df1d4f173b72, gate G1 passed) instead of re-running
# generation + human review on Kaggle. If the import fails, the data-stage cell
# below runs WITHOUT --auto-review and the G1 gate enforces human verification.
import json
import os
from pathlib import Path

from huggingface_hub import snapshot_download

DATA_REPO = os.environ.get('CG_DATA_REPO', 'dubeydevansh/circuitgraft-phase4-data')
summary_path = Path('artifacts/data/summary.json')

if not summary_path.exists():
    try:
        snapshot_download(
            repo_id=DATA_REPO,
            repo_type='dataset',
            local_dir='artifacts/data',
            token=os.environ['HF_TOKEN'],
        )
        print('Imported curated data snapshot from', DATA_REPO)
    except Exception as exc:
        print(f'WARN: snapshot import failed ({type(exc).__name__}: {exc})')
        print('Falling back to the data-stage cell; the G1 gate below will enforce human review.')

if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print('data run_id:', summary.get('run_id'))
    print('G1:', summary.get('gate_g1', {}).get('status'))
    assert Path('artifacts/data/medmcqa_sft/train.jsonl').exists(), 'SFT rows missing'
    assert Path('artifacts/data/pairs/safety.jsonl').exists(), 'pairs missing'
    print('data snapshot ready; the data-stage cell will skip automatically.')


In [ ]:
%%bash
export TF_CPP_MIN_LOG_LEVEL=3
export ABSL_LOGGING_MIN_SEVERITY=3
export TF_ENABLE_ONEDNN_OPTS=0
export PYTHONUNBUFFERED=1
set -euo pipefail

# Skipped when the verified data snapshot was imported in the previous cell.
if [ -f artifacts/data/summary.json ]; then
  echo 'SKIP data stage: verified snapshot already present (artifacts/data/summary.json)'
else
  python -m circuitgraft.cli data --config configs/data.yaml
fi


In [ ]:
# Gate G1: all configured sets must exist and domain-sensitive rows must be
# human-verified in the snapshot. No auto-sealing: if G1 is not passed the run
# stops here instead of fabricating review labels.
import json
from pathlib import Path

summary_path = Path('artifacts/data/summary.json')
if not summary_path.exists():
    raise FileNotFoundError('data stage did not write artifacts/data/summary.json')
summary = json.loads(summary_path.read_text(encoding='utf-8'))

print('Evaluation sets:')
for name, info in summary.get('eval', {}).items():
    print(
        f"  {name:22s} {str(info.get('status')):32s} "
        f"rows={info.get('rows')} human_verified={info.get('human_verified')}"
    )
for kind in ('benign', 'harmful'):
    info = summary.get('generation', {}).get(f'domain_{kind}', {})
    if info.get('review_path'):
        print(f"  review {kind}: {info['review_path']}")

gate = summary.get('gate_g1', {})
print('\nG1:', gate.get('status'))
for reason in gate.get('reasons', []):
    print('  -', reason)

if gate.get('status') != 'passed':
    raise RuntimeError(
        'Data gate G1 has not passed. Do not proceed: complete the human-review '
        'CSVs with real review labels, then re-run the data cells.'
    )
print('\nG1 passed; proceeding to drift.')


## 7. Stage 2 (Drift SFT) & Stage 3 (Baseline Audit)

Trains $\theta_D$ (drifted checkpoint) and runs baseline safety audit comparing $\theta_A$ vs $\theta_D$.

In [ ]:
%%bash
export TF_CPP_MIN_LOG_LEVEL=3
export ABSL_LOGGING_MIN_SEVERITY=3
export TF_ENABLE_ONEDNN_OPTS=0
export PYTHONUNBUFFERED=1

set -euo pipefail
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

# Cloud drift profile: configs/drift_t4.yaml = micro-batch 1 x grad_accum 32
# (effective batch 32 -> identical LR schedule and 314 optimizer steps as the
# local run). This is the exact shape validated on the 8 GB RX 6600 (zero OOM
# across the whole run); micro-batch 4 OOM'd twice on the 14.56 GiB T4 at the
# fp32 CE upcast over the 262144-token vocab (4.29 GiB buffer at seq 1024).
DRIFT_CONFIG=configs/drift_t4.yaml
echo "drift config: $DRIFT_CONFIG"
python - <<'PYEOF'
import yaml
cfg = yaml.safe_load(open('configs/drift_t4.yaml'))['train']
print('effective batch:', cfg['batch_size'], 'x', cfg['grad_accum'],
      '=', cfg['batch_size'] * cfg['grad_accum'])
PYEOF
nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
python -m circuitgraft.cli drift --config "$DRIFT_CONFIG"
python -m circuitgraft.cli audit --config configs/eval.yaml --models aligned,drifted


## 8. Stage 4 (Dual-Circuit Discovery)

`configs/discover.yaml` carries the logged prereg deviation for the 8 GB card:
`dtype: float16`, `max_seq_len: 96`, `batch_size: 1` (pairs are short: safety
max 81 tokens; domain median 48, max 206). The runner also reties
`unembed.W_U` to `embed.W_E.T` when `logit_scale == 1.0`, which frees a
duplicate ~1 GiB untied LM head on load (see `_retie_unembed` in
`circuitkit_runner.py`). On a 15 GB T4/P100 this shape runs with large
headroom; `ig_steps: 5` x 256 pairs is the main time cost. If it still OOMs,
lower `n_examples` or use the P100 runtime.


In [ ]:
%%bash
export TF_CPP_MIN_LOG_LEVEL=3
export ABSL_LOGGING_MIN_SEVERITY=3
export TF_ENABLE_ONEDNN_OPTS=0
export PYTHONUNBUFFERED=1
set -euo pipefail

python -m circuitgraft.cli discover --config configs/discover.yaml


## 9. Stage 5 (Baselines) & Stage 6 (Circuit Grafting)

Runs `cg_project`, `cg_uniform`, `cg_exclude`, `cg_distributed`, and `cg_random`. `cg_fisher` runs only if its Fisher artifact exists; otherwise it is skipped as the declared scope cut.

In [ ]:
%%bash
export TF_CPP_MIN_LOG_LEVEL=3
export ABSL_LOGGING_MIN_SEVERITY=3
export TF_ENABLE_ONEDNN_OPTS=0
export PYTHONUNBUFFERED=1
set -euo pipefail

python -m circuitgraft.cli baselines --config configs/baselines/resta.yaml
python -m circuitgraft.cli baselines --config configs/baselines/wiseft.yaml
python -m circuitgraft.cli baselines --config configs/baselines/ctheta.yaml
python -m circuitgraft.cli baselines --config configs/baselines/safelora.yaml
python -m circuitgraft.cli graft --config configs/graft/cg_project.yaml
python -m circuitgraft.cli graft --config configs/graft/cg_uniform.yaml
python -m circuitgraft.cli graft --config configs/graft/cg_exclude.yaml
python -m circuitgraft.cli graft --config configs/graft/cg_distributed.yaml
python -m circuitgraft.cli graft --config configs/graft/cg_random.yaml

# cg_fisher is the first allowed scope cut (SPEC.md). Run it only when
# the domain Fisher artifact exists; otherwise leave it out and disclose it.
if [[ -f artifacts/graft/fisher/domain_fisher.safetensors ]]; then
    python -m circuitgraft.cli graft --config configs/graft/cg_fisher.yaml
else
    echo 'SKIP cg_fisher: artifacts/graft/fisher/domain_fisher.safetensors is absent (declared scope cut)'
fi


## 10. Stage 7 (Comprehensive Evaluation) & Stage 8 (Attacks)

In [ ]:
%%bash
export TF_CPP_MIN_LOG_LEVEL=3
export ABSL_LOGGING_MIN_SEVERITY=3
export TF_ENABLE_ONEDNN_OPTS=0
export PYTHONUNBUFFERED=1
set -euo pipefail

python -m circuitgraft.cli eval --config configs/eval.yaml --all
python -m circuitgraft.cli attack --config configs/attacks.yaml --type abliterate
python -m circuitgraft.cli eval-attacks --config configs/eval.yaml


## 11. Stage 9 (Reporting & README) & Stage 10 (Publish to HF)

In [ ]:
%%bash
export TF_CPP_MIN_LOG_LEVEL=3
export ABSL_LOGGING_MIN_SEVERITY=3
export TF_ENABLE_ONEDNN_OPTS=0
export PYTHONUNBUFFERED=1
set -euo pipefail

python -m circuitgraft.cli report --output-dir results/s9 --readme README.md
python -m circuitgraft.cli publish
